# Library Imports 

In [58]:
import pickle
import os
from pathlib import Path
from collections import defaultdict

from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.metrics import accuracy_score 

from behave_analysis.process.session import get_experiment
from behave_analysis.analyze.overview.homing_analysis.decoding_spatial_eff.model import compute_accuracy_data, plot_accuracy_across_sessions

# Data Imports

In [59]:
from behave_analysis.database.Experiments.JAL003_ex import JAL3_25aug, JAL3_1sept, JAL3_4sept, JAL3_7sept
from behave_analysis.database.Experiments.JAL004_ex import JAL4_3rdSept, JAL4_19thSept, JAL4_28aug, JAL4_11thSept
from behave_analysis.database.Experiments.JAL005_ex import JAL005_8thSept, JAL005_21stSept
from behave_analysis.database.Experiments.JAL006_ex import JAL6_28mar, JAL6_flip4_21mar, JAL6_flip5_25mar, JAL6_flip3_18mar, JAL6_flip7_1apr
from behave_analysis.database.Experiments.JAL007_ex import JAL7_sesh8_9apr, JAL7_sesh9_16apr, JAL7_flip5_22mar, JAL7_flip2_12mar, JAL7_23apr,JAL7_30apr
from behave_analysis.database.Experiments.JAL008_ex import JAL8_flip1_25apr, JAL8_flip2_29apr, JAL8_tiny_3may, JAL8_flip4_10may, JAL8_14may, JAL8_21may, JAL8_flip3_7may

# Defined session names, experiments and hash mice to sessions

In [60]:
# The values must be in order of data for the code to work
mice_groups = {
    "JAL6": ['JAL6_flip3_18mar', 'JAL6_flip4_21mar', 'JAL6_flip5_25mar', 'JAL6_28mar'],
    "JAL3": ['JAL3_25aug', 'JAL3_1sept', 'JAL3_4sept', 'JAL3_7sept'],
    "JAL7": ['JAL7_flip2_12mar', 'JAL7_flip5_22mar', 'JAL7_sesh8_9apr', 'JAL7_sesh9_16apr', 'JAL7_23apr'],
    "JAL8": ['JAL8_flip1_25apr', 'JAL8_flip2_29apr', 'JAL8_flip4_10may', "JAL8_flip3_7may", 'JAL8_14may'],
    "JAL4": ['JAL4_28aug', 'JAL4_3rdSept', 'JAL4_11thSept', 'JAL4_19thSept'],
    "JAL5": ['JAL5_8thSept', 'JAL5_21stSept']}

# The experiment objects index must match the session name index for the code to work
experiments_objects = [JAL6_flip3_18mar, JAL6_flip4_21mar, JAL6_flip5_25mar, JAL6_28mar,
                       JAL3_25aug, JAL3_1sept, JAL3_4sept, JAL3_7sept,
                       JAL005_8thSept, JAL005_21stSept,
                       JAL7_sesh8_9apr, JAL7_sesh9_16apr, JAL7_flip5_22mar, JAL7_flip2_12mar, JAL7_23apr,
                       JAL8_flip1_25apr, JAL8_flip2_29apr, JAL8_flip4_10may, JAL8_14may, JAL8_flip3_7may,
                       JAL4_3rdSept, JAL4_19thSept, JAL4_28aug, JAL4_11thSept]

# The session name indexes must match the experiment object indexes
session_names = ["JAL6_flip3_18mar", "JAL6_flip4_21mar", "JAL6_flip5_25mar", "JAL6_28mar",
                 "JAL3_25aug", "JAL3_1sept", "JAL3_4sept", "JAL3_7sept",
                 "JAL5_8thSept", "JAL5_21stSept",
                 "JAL7_sesh8_9apr", "JAL7_sesh9_16apr", "JAL7_flip5_22mar", "JAL7_flip2_12mar", "JAL7_23apr",
                 "JAL8_flip1_25apr", "JAL8_flip2_29apr", "JAL8_flip4_10may", "JAL8_14may", "JAL8_flip3_7may",
                 "JAL4_3rdSept", "JAL4_19thSept", "JAL4_28aug", "JAL4_11thSept"]

# Load the data

In [62]:
# This cell is only required if you wish to re-generate the accuracy data below. Else if you want to genreate data for the model you will need to go into the data generation script. As this cell loads pre-generated data. 

# dir = Path("Z:\Jasmine_Laurence\single_trial_overview\decoding_spatial_efficiency")
# name_of_data = "compute_the_first_20_frames_before_homing"
# path = dir / f"{name_of_data}.pkl"
# with open(path, "rb") as f:
#     compute_the_first_20_frames_before_homing =  pickle.load(f)

# Generate the accuracy data

In [ ]:
# If the accuracy data does not exist, create it. Or if you want to run a control test or another window of data.
# Else, you will load the accuracy data below.

# compute_accuracy_data(
#         data_type=compute_the_first_20_frames_before_homing, 
#         experiments_objects= experiments_objects, 
#         data_type_name="500ms_before_accuracy", 
#         random_labels=False
#     )

# Load prior accuracy and coefficients

In [63]:
with open(r"Z:\Jasmine_Laurence\single_trial_overview\decoding_spatial_efficiency\500ms_before_accuracy\500ms_before_accuracy_accuracy_ks_coeffs_data.pkl", "rb") as f:
    accuracy_ks_coeffs_data = pickle.load(f)

In [ ]:
accuracy_ks_coeffs_data.keys()
coefs_unotuched = accuracy_ks_coeffs_data["coefs"]
accuracy_data = accuracy_ks_coeffs_data["accuracy_data"]

# Average the coefficients

In [66]:
# Step 1: Average coefficients across folds for each session
average_coefs = {}
for session_name, folds in coefs_unotuched.items():
    # Stack the coefficients from all folds and compute the mean
    coefficients = np.array(list(folds.values()))  # Shape: (num_folds, num_features)
    average_coefs[session_name] = coefficients.mean(axis=0)  # Average across folds

In [ ]:
print(f"The len of the average_coefs is {len(average_coefs)}")

# Load head direction cells

In [68]:
with open(r"Z:\Jasmine_Laurence\single_trial_overview\decoding_spatial_efficiency\head_direction_cells.pkl", "rb") as f:
    head_direction = pickle.load(f)

# Check how many high coeffcients are head direction

Max absolute coefficients

In [ ]:

# Initialize counters and data collection for plotting
cells_with_top_20_coefficients = 0
cells_with_top_20_coefficients_that_are_hdir_cells = 0
session_data = []
coeff_counts = defaultdict(defaultdict)

# Exlcude sessions because of missmatch of cells
excluded_sessions = ["JAL6_flip5_25mar", "JAL8_flip4_10may", "JAL4_19thSept"]

for experiment, session_name in zip(experiments_objects, session_names):
    
    if session_name in excluded_sessions:
        print(f"Excluding session: {session_name}")
        continue
    
    if session_name not in accuracy_data.keys():
        print(f"Session: {session_name} not in accuracy data")
        continue
    
    print(f"Loading data for session: {session_name}")
    loaded_session = get_experiment(experiment)

    # Set paths
    base_path = loaded_session.base_path
    processed_path = loaded_session.processed_path
    good_cluster_path = os.path.join(base_path, processed_path, "good_cluster_ids.npy")

    # Load data
    hdir_cells = np.array(head_direction[session_name])
    accuracy = accuracy_data[session_name]
    coefs_ = average_coefs[session_name]
    good_cluster_ids = np.load(good_cluster_path)
    assert len(good_cluster_ids) == len(coefs_), "The number of good cluster ids and coefficients do not match"

    # Convert coefficients to a list if not already
    if isinstance(coefs_, defaultdict):
        coefs_ = list(coefs_.values())

    # Ensure coefficients are converted to a numpy array
    coefs_ = np.array(coefs_)

    # Get the indices of the top 20 coefficients
    top_20_indices = np.argsort(np.abs(coefs_))[-20:]
    top_20_coefficients = coefs_[top_20_indices]
    assert max(coefs_) in top_20_coefficients, "The max coefficient is not in the top 20 coefficients"
    good_cluster_ids_top_20 = good_cluster_ids[top_20_indices] # Get the good cluster ids of the top 20 coefficients
    count_good_cluster_ids_top_20 = len(good_cluster_ids_top_20)
    
    # Get the head direction cells that are in the top 20 coefficients
    hdir_cells_top_20 = np.intersect1d(hdir_cells, good_cluster_ids_top_20)
    
    # Fill data
    coeff_counts[session_name]["#_head_direction_cells"] = len(hdir_cells)
    coeff_counts[session_name]["#_hdir_cells_that_top_20"] = len(hdir_cells_top_20)

In [ ]:
coeff_counts

In [ ]:
data = coeff_counts

# Prepare data for plotting
sessions = list(data.keys())
hd_cells = [data[session]['#_head_direction_cells'] for session in sessions]
top_20_hd_cells = [data[session]['#_hdir_cells_that_top_20'] for session in sessions]
accuracies = [accuracy_data[session] for session in sessions]

# Calculate averages
average_hd_cells = np.mean(hd_cells)
average_top_20_hd_cells = np.mean(top_20_hd_cells)

# Bar positions
x_positions = np.arange(len(sessions))

# Plotting
fig, ax1 = plt.subplots(figsize=(12, 6))

# Plot bars on the primary y-axis
bar_width = 0.4
bar1 = ax1.bar(x_positions - bar_width / 2, hd_cells, bar_width, label='Head Direction Cells')
bar2 = ax1.bar(x_positions + bar_width / 2, top_20_hd_cells, bar_width, label='Top 20 coefficients intersecting with HD Cells')

# Plot points and lines on the primary y-axis
ax1.plot(x_positions - bar_width / 2, hd_cells, 'o-', color='blue')
ax1.plot(x_positions + bar_width / 2, top_20_hd_cells, 'o-', color='orange')

# Average bars on the primary y-axis
ax1.bar(len(sessions), average_hd_cells, bar_width, label='Average HD Cells', color='blue', alpha=0.5)
ax1.bar(len(sessions) + 1, average_top_20_hd_cells, bar_width, label='Average Intersect', color='orange', alpha=0.5)

# Secondary y-axis for accuracy
ax2 = ax1.twinx()
ax2.plot(x_positions, accuracies, color='gray', label='Accuracy')

# Customizing the primary y-axis
ax1.set_xlabel('Sessions')
ax1.set_ylabel('Number of Cells')
ax1.set_title('How many of the top 20 coefficients per session in the logistic regression are head direction cells?')
ax1.set_xticks(list(x_positions) + [len(sessions), len(sessions) + 1])
ax1.set_xticklabels(sessions + ['Avg HD', 'Avg Intersect'], rotation=45, ha='right')
ax1.legend(loc='upper left')

# Customizing the secondary y-axis
ax2.set_ylabel('Accuracy of logistic regression')
ax2.legend(loc='upper right')

plt.tight_layout()
plt.show()

In [ ]:
# Plot the results
categories = ['Cells with Top 20 Coefficients', 'Head Direction Cells in Top 20 Coefficients']
counts = [cells_with_top_20_coefficients, cells_with_top_20_coefficients_that_are_hdir_cells]

plt.figure(figsize=(10, 8))
plt.bar(categories, counts, color='lightblue', edgecolor='black')

# Now select the 500ms before the homing

In [ ]:
plot_accuracy_across_sessions(accuracy_data, mice_groups=mice_groups, plot_title="500ms before homing")

# Check out coefficients

In [ ]:
# Step 1: Average coefficients across folds for each session
average_coefs = {}
for session_name, folds in coefs.items():
    # Stack the coefficients from all folds and compute the mean
    coefficients = np.array(list(folds.values()))  # Shape: (num_folds, num_features)
    average_coefs[session_name] = coefficients.mean(axis=0)  # Average across folds

# Step 2: Plot coefficients for each session
for session_name, session_coefs in average_coefs.items():
    accuracy = accuracy_data[session_name]
    # Rank coefficients from largest to smallest
    sorted_indices = np.argsort(np.abs(session_coefs))[::-1]  # Sort by absolute value, descending
    sorted_coefs = session_coefs[sorted_indices]

    # Plot
    plt.figure(figsize=(10, 6))
    plt.bar(range(len(sorted_coefs)), sorted_coefs, tick_label=sorted_indices)
    plt.title(f"Ranked Coefficients for Session: {session_name}, with Accuracy: {accuracy}")
    plt.xlabel("Feature Index (Ranked)")
    plt.ylabel("Coefficient Value")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
_14th_may_coefs = average_coefs["JAL8_14may"]
_7th_may_coefs = average_coefs["JAL8_flip3_7may"]

# Load the followng numpy arrau
_14_file = np.load(r"W:\branco\Laurence\JAL008\JAL008_shelter_barrier_flip_5_2024_05_14T10_18_03\processed_data\good_cluster_Ids.npy")
_7_file = np.load(r"W:\branco\Laurence\JAL008\JAL008_shelter_barrier_flip_3_2024_05_07T10_16_26\processed_data\good_cluster_Ids.npy")

# Acess the top positive 10 coefficients for the 14th may
_14th_may_indices = np.argsort(_14th_may_coefs)[::-1][:10]

print("The top 10 positive coefficients for the 14th may are", _14th_may_indices)
print("The top 10 positive coefficients for the 14th may are", _14th_may_coefs[_14th_may_indices])
print("The cell IDs for the 14th may are", sorted(_14_file[_14th_may_indices]))

# Acess the top positive 10 coefficients for the 7th may
_7th_may_indices = np.argsort(_7th_may_coefs)[::-1][:10]

print("The top 10 positive coefficients for the 7th may are", _7th_may_indices)
print("The top 10 positive coefficients for the 7th may are", _7th_may_coefs[_7th_may_indices])
print("The cell IDs for the 7th may are", sorted(_7_file[_7th_may_indices]))


# Plot the KS vs decoding accuracy analysis

In [ ]:
# Extract average, max KS stats, and accuracies
def extract_ks_and_accuracy(data):
    avg_ks_values = []
    max_ks_values = []
    accuracies = []

    for session, trials in data.items():
        for trial, metrics in trials.items():
            if all(k in metrics for k in ['x', 'y', 'speed', 'hdir', 'accuracy']):
                ks_values = [metrics['x'], metrics['y'], metrics['speed'], metrics['hdir']]
                avg_ks_values.append(np.mean(ks_values))
                max_ks_values.append(np.max(ks_values))
                accuracies.append(metrics['accuracy'])

    return np.array(avg_ks_values), np.array(max_ks_values), np.array(accuracies)

# Perform the correlation analysis
def analyze_correlation(ks_values, accuracies):
    correlation = np.corrcoef(ks_values, accuracies)[0, 1]
    return correlation

# Extract data
avg_ks_values, max_ks_values, accuracies = extract_ks_and_accuracy(_500ms_before_ks_decoding_comparison)

# Check for sufficient data
if len(avg_ks_values) > 0 and len(max_ks_values) > 0 and len(accuracies) > 0:
    # Correlations
    avg_correlation = analyze_correlation(avg_ks_values, accuracies)
    max_correlation = analyze_correlation(max_ks_values, accuracies)

    print(f"Correlation between average KS statistic and decoding accuracy: {avg_correlation}")
    print(f"Correlation between max KS statistic and decoding accuracy: {max_correlation}")

    # Plot correlations across all sessions and folds
    plt.figure()
    plt.scatter(avg_ks_values, accuracies, alpha=0.7, label='Average KS')
    plt.scatter(max_ks_values, accuracies, alpha=0.7, label='Max KS', color='r')
    plt.xlabel('KS Statistic')
    plt.title(f"Max and Average KS Statistic vs Decoding Accuracy (Average r={avg_correlation:.2f}, Max r={max_correlation:.2f})")
    plt.legend()
    plt.show()

    # Plot individual KS variables against decoding accuracy
    ks_variables = ['x', 'y', 'speed', 'hdir']
    fig, axs = plt.subplots(2, 2, figsize=(10, 10))
    axs = axs.ravel()

    for i, var in enumerate(ks_variables):
        ks_values = []
        accuracies = []
        for session, trials in _500ms_before_ks_decoding_comparison.items():
            for trial, metrics in trials.items():
                if var in metrics and 'accuracy' in metrics:
                    ks_values.append(metrics[var])
                    accuracies.append(metrics['accuracy'])

        correlation = analyze_correlation(ks_values, accuracies)
        axs[i].scatter(ks_values, accuracies, alpha=0.7)
        axs[i].set_title(f'{var} vs Accuracy (r={correlation:.2f})')
        axs[i].set_xlabel("KS Statistic")
        axs[i].set_ylabel('Accuracy')

    plt.tight_layout()
    plt.show()
else:
    print("Insufficient data for analysis.")